In [2]:
from google.colab import drive
import pandas as pd
import numpy as np

drive.mount('/content/drive')

path_sandbox = '/content/drive/MyDrive/datasets/DDoS_Sandbox_Dataset/train_sandbox_v3.csv'
path_cic = '/content/drive/MyDrive/datasets/CIC/Wednesday-workingHours.pcap_ISCX.csv'
path_dos1_thursday = '/content/drive/MyDrive/datasets/CSE-CIC-IDS2018/DoS1-Thursday-15-02-2018_TrafficForML_CICFlowMeter.parquet'
path_dos2_friday = '/content/drive/MyDrive/datasets/CSE-CIC-IDS2018/DoS2-Friday-16-02-2018_TrafficForML_CICFlowMeter.parquet'

CORE_FEATURES = {
    'flow_duration': [],
    'flow_iat_mean': [],
    'flow_iat_std':  [],
    'flow_iat_max':  [],
    'fwd_iat_mean': [],
    'fwd_iat_std':  [],
    'fwd_iat_max':  [],
    'bwd_iat_mean': [],
    'bwd_iat_std':  [],
    'bwd_iat_max':  [],
    'flow_iat_min': [],
    'fwd_iat_min':  [],
    'bwd_iat_min':  [],

    'flow_byts_s': ['flow_bytes_s', 'flow_bytes_sec'],
    'flow_pkts_s': ['flow_packets_s', 'flow_packets_sec'],
    'fwd_pkts_s': ['fwd_packets_s', 'fwd_packets_sec'],
    'bwd_pkts_s': ['bwd_packets_s', 'bwd_packets_sec'],

    'active_mean': [],
    'active_std':  [],
    'active_max':  [],
    'active_min':  [],

    'idle_mean': [],
    'idle_std':  [],
    'idle_max':  [],
    'idle_min':  [],

    'syn_flag_cnt': ['syn_flag_count'],
    'rst_flag_cnt': ['rst_flag_count'],

    'tot_fwd_pkts': ['total_fwd_packets'],
    'tot_bwd_pkts': ['total_backward_packets'],

    'init_fwd_win_byts': ['init_win_bytes_forward', 'init_fwd_win_bytes'],
    'init_bwd_win_byts': ['init_win_bytes_backward', 'init_bwd_win_bytes'],
}


TIME_COLS = [
    'flow_duration',
    'flow_iat_mean', 'flow_iat_std', 'flow_iat_max', 'flow_iat_min',
    'fwd_iat_mean', 'fwd_iat_std', 'fwd_iat_max', 'fwd_iat_min',
    'bwd_iat_mean', 'bwd_iat_std', 'bwd_iat_max', 'bwd_iat_min',
    'active_mean', 'active_std', 'active_max', 'active_min',
    'idle_mean', 'idle_std', 'idle_max', 'idle_min',
]

LABEL_MAP = {
      'normal': 'BENIGN',
      'benign': 'BENIGN',
    'ad_slow': 'Slowloris',
      'slowloris': 'Slowloris',
      'dos slowloris': 'Slowloris',
    'dos slowhttptest': 'Slowhttptest',
    'dos attacks-slowloris': 'Slowloris',
    'dos attacks-slowhttptest': 'Slowhttptest',
}



def prepare(path, label_col, target_values, source_name, file_type='csv', seconds_to_micro=False):
    if file_type == 'parquet':
        df = pd.read_parquet(path)
    else:
        df = pd.read_csv(path, low_memory=False)

    df.columns = df.columns.str.strip()
    df[label_col] = df[label_col].astype(str).str.strip()
    df = df[df[label_col].isin(target_values)].copy()

    df['label'] = df[label_col].str.lower().map(LABEL_MAP)
    df = df.dropna(subset=['label'])
    df = df.drop(columns=[label_col])

    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(r'[^\w]+', '_', regex=True).str.strip('_'))
    df = df.loc[:, ~df.columns.duplicated()]

    for canonical, variants in CORE_FEATURES.items():
        if canonical not in df.columns:
            for v in variants:
                if v in df.columns:
                    df = df.rename(columns={v: canonical})
                    break

    if seconds_to_micro:
        present = [c for c in TIME_COLS if c in df.columns]
        df[present] = df[present] * 1_000_000

    df['dataset_source'] = source_name
    return df


df1 = prepare(path_sandbox, 'Label', ['BENIGN', 'Slowloris', 'DoS slowloris'], 'Sandbox')
df2 = prepare(path_cic, 'Label', ['BENIGN', 'DoS slowloris', 'DoS Slowhttptest'], 'CIC_Wednesday')
df3 = prepare(path_dos1_thursday, 'Label', ['Benign', 'DoS attacks-Slowloris'], 'CSE_DoS1_Thursday', file_type='parquet')
df4 = prepare(path_dos2_friday, 'Label', ['Benign', 'DoS attacks-SlowHTTPTest'], 'CSE_DoS2_Friday', file_type='parquet')

common_features = [
    c for c in CORE_FEATURES
    if c in df1.columns and c in df2.columns and c in df3.columns and c in df4.columns
]
final_cols = common_features + ['label', 'dataset_source']

unified_dataset = pd.concat(
    [df1[final_cols], df2[final_cols], df3[final_cols], df4[final_cols]],
    ignore_index=True
)

unified_dataset[common_features] = unified_dataset[common_features].replace([-1, np.inf, -np.inf], np.nan)
for col in common_features:
    unified_dataset[col] = unified_dataset.groupby('dataset_source')[col].transform(
        lambda s: s.fillna(s.median())
    )

print(f"\nعدد الميزات المشتركة النهائية: {len(common_features)} / {len(CORE_FEATURES)}")
print(common_features)
print(f"\nشكل الداتاسيت النهائي: {unified_dataset.shape}")
print(unified_dataset['dataset_source'].value_counts())
print(unified_dataset['label'].value_counts())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

عدد الميزات المشتركة النهائية: 31 / 31
['flow_duration', 'flow_iat_mean', 'flow_iat_std', 'flow_iat_max', 'fwd_iat_mean', 'fwd_iat_std', 'fwd_iat_max', 'bwd_iat_mean', 'bwd_iat_std', 'bwd_iat_max', 'flow_iat_min', 'fwd_iat_min', 'bwd_iat_min', 'flow_byts_s', 'flow_pkts_s', 'fwd_pkts_s', 'bwd_pkts_s', 'active_mean', 'active_std', 'active_max', 'active_min', 'idle_mean', 'idle_std', 'idle_max', 'idle_min', 'syn_flag_cnt', 'rst_flag_cnt', 'tot_fwd_pkts', 'tot_bwd_pkts', 'init_fwd_win_byts', 'init_bwd_win_byts']

شكل الداتاسيت النهائي: (2150746, 33)
dataset_source
CSE_DoS1_Thursday    753406
Sandbox              499340
CIC_Wednesday        451326
CSE_DoS2_Friday      446674
Name: count, dtype: int64
label
BENIGN          2000771
Slowloris        144421
Slowhttptest       5554
Name: count, dtype: int64
